# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.4360675  -0.42460766  0.54240985  0.84032762 -0.12609643]
 [ 0.50947869 -0.28049692  0.75230332  0.88998279 -0.41735652]
 [-0.06692992 -0.67239729  0.55613167 -0.24541905  0.33521254]
 [-0.63081821  0.60249351  0.37460334  0.177017   -0.05016585]
 [-0.68111612 -0.62449323 -0.22825243  0.89448557  0.69527399]
 [-0.39445698  0.89172712  0.17905665  0.71040451  0.08293314]
 [-0.77220675 -0.19272046 -0.73395549 -0.83133909  0.17561734]
 [ 0.15258569 -0.37528884  0.12428065 -0.72046702  0.51286643]
 [-0.06916997 -0.46894142 -0.21721944 -0.87571919  0.77016077]
 [ 0.93265693 -0.01661347 -0.01844189  0.37417589  0.72900784]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a1', 'a1', 'a1', 'a2', 'a2', 'a1', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 0, 1, 1, 0, 0, 0, 0, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.09s/it]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.09s/it, loss=2540.4917]

SVI:   6%|▌         | 2/34 [00:01<00:34,  1.09s/it, loss=2518.8684]

SVI:   9%|▉         | 3/34 [00:01<00:33,  1.09s/it, loss=1836.4865]

...

SVI:  84%|████████▍ | 21/25 [00:00<00:03,  1.10it/s, loss=3361.3103]

SVI:  88%|████████▊ | 22/25 [00:00<00:02,  1.10it/s, loss=3255.2532]

SVI:  92%|█████████▏| 23/25 [00:00<00:01,  1.10it/s, loss=2928.6365]

SVI:  96%|█████████▌| 24/25 [00:00<00:00,  1.10it/s, loss=2757.0701]

SVI: 100%|██████████| 25/25 [00:00<00:00,  1.10it/s, loss=2546.2295]